# 🗂️ Hash Tables -- Runnable Notebook

Companion to [`README.md`](README.md).

A tiny hash table built from scratch (hashing + chaining), then Python's `dict`/`set`/`Counter`, and the classic uses.

## 1. A hash table from scratch (separate chaining)
Each bucket is a small list of `(key, value)` pairs; collisions just mean "append to this bucket's list".

In [ ]:
class SimpleHashTable:
    def __init__(self, capacity=8):
        self.capacity = capacity
        self.buckets = [[] for _ in range(capacity)]   # chaining: one list per bucket
        self.size = 0

    def _index(self, key):
        return hash(key) % self.capacity                # O(1): compute, don't search

    def put(self, key, value):
        i = self._index(key)
        bucket = self.buckets[i]
        for j, (k, _) in enumerate(bucket):
            if k == key:
                bucket[j] = (key, value)                 # update existing key
                return
        bucket.append((key, value))                      # new key
        self.size += 1
        if self.size / self.capacity > 0.75:             # load factor too high -> resize
            self._resize()

    def get(self, key, default=None):
        bucket = self.buckets[self._index(key)]
        for k, v in bucket:
            if k == key:
                return v
        return default

    def _resize(self):
        old_buckets = self.buckets
        self.capacity *= 2                                # double capacity
        self.buckets = [[] for _ in range(self.capacity)]
        self.size = 0
        for bucket in old_buckets:                        # re-hash every existing key
            for k, v in bucket:
                self.put(k, v)


ht = SimpleHashTable(capacity=4)
ht.put("alice", 30)
ht.put("bob", 25)
ht.put("carol", 40)
print("alice ->", ht.get("alice"))
print("missing key ->", ht.get("dave", "not found"))
assert ht.get("alice") == 30
assert ht.get("dave", "not found") == "not found"

# Force enough inserts to trigger a resize, and confirm everything survives it
for i in range(20):
    ht.put(f"key{i}", i)
assert ht.capacity > 4                                    # it grew
assert ht.get("alice") == 30                                # old data survived the resize
assert ht.get("key15") == 15
print("capacity after growth:", ht.capacity)

### Step-by-step: what this cell actually does

`SimpleHashTable` implements separate chaining: `self.buckets` is a fixed-size list of lists, and `_index(key)` maps a key to a bucket via `hash(key) % self.capacity`. `put` and `get` both start by computing that same index, then linearly scan the bucket list — `put` either overwrites a matching key or appends a new `(key, value)` pair (this linear scan/append *is* the collision handling: two different keys that land in the same bucket just live side by side in that bucket's list), and `get` scans the same list looking for a match. `put` also checks the load factor (`size / capacity`) after every insert and calls `_resize()` once it exceeds `0.75`, which doubles `capacity`, rebuilds empty buckets, and re-`put`s every existing entry so it lands in its new bucket. The demo builds `ht = SimpleHashTable(capacity=4)`, inserts `"alice"`, `"bob"`, `"carol"`, confirms two lookups (`ht.get("alice") == 30`, `ht.get("dave", "not found") == "not found"`), then inserts 20 more keys (`"key0"`..`"key19"`) to force three resizes, and finally asserts `ht.capacity > 4` and that both an old key (`"alice"`) and a key added after growth (`"key15"`) still resolve correctly. Bucket indices below come from one real run of this exact code (`hash()` on strings is seeded per-process, so the *specific* numbers will differ on another run/seed — the mechanics traced here do not).

#### `put(key, value)` — insert or update, chain on collision, resize on overload

1. Compute `i = self._index(key)` — the only place a key touches the array.
2. Linearly scan `self.buckets[i]`; if an entry with the same key exists, overwrite its value in place and return (no growth).
3. Otherwise this is a new key: append `(key, value)` to `self.buckets[i]` — if another key already lives there, this *is* the collision, resolved by growing that bucket's list — and increment `self.size`.
4. If `self.size / self.capacity > 0.75`, call `self._resize()` (doubles `capacity`, rehashes everything).

Demo: `ht = SimpleHashTable(capacity=4)`, then `ht.put("alice",30)`, `ht.put("bob",25)`, `ht.put("carol",40)`, then `for i in range(20): ht.put(f"key{i}", i)`.

| Step | Action | `bucket[i]` after | `size` after | `capacity` after |
|---|---|---|---|---|
| 1 | `put("alice", 30)`: `i=1`, empty bucket → **APPEND** new key | `[('alice', 30)]` | 1 | 4 |
| 2 | `put("bob", 25)`: `i=2`, empty bucket → **APPEND** new key | `[('bob', 25)]` | 2 | 4 |
| 3 | `put("carol", 40)`: `i=2` — same bucket as `"bob"` → **COLLISION**, append alongside | `[('bob', 25), ('carol', 40)]` | 3 | 4 |
| 4 | `put("key0", 0)`: `i=1` — same bucket as `"alice"` → **COLLISION**, append; `size=4`, `4/4=1.0 > 0.75` → **RESIZE** to 8 (see `_resize` below) | `[('alice', 30), ('key0', 0)]` (pre-resize) | 4 | 8 |
| 5 | `put("key1", 1)`: `i=7` (cap 8), empty bucket → **APPEND** | `[('key1', 1)]` | 5 | 8 |
| 6 | `put("key2", 2)`: `i=0`, empty bucket → **APPEND** | `[('key2', 2)]` | 6 | 8 |
| 7 | `put("key3", 3)`: `i=5` — same bucket as `"key0"` → **COLLISION**, append; `size=7`, `7/8=0.875 > 0.75` → **RESIZE** to 16 | `[('key0', 0), ('key3', 3)]` (pre-resize) | 7 | 16 |
| 8 | `put("key4", 4)`: `i=2` (cap 16) — same bucket as `"carol"` → **COLLISION**, append | `[('carol', 40), ('key4', 4)]` | 8 | 16 |
| 9 | `put("key5", 5)`: `i=8` — same bucket as `"key2"` → **COLLISION**, append | `[('key2', 2), ('key5', 5)]` | 9 | 16 |
| 10 | `put("key6", 6)`: `i=8` — same bucket → **COLLISION**, append | `[('key2', 2), ('key5', 5), ('key6', 6)]` | 10 | 16 |
| 11 | `put("key7", 7)`: `i=8` — same bucket → **COLLISION**, append | `[('key2', 2), ('key5', 5), ('key6', 6), ('key7', 7)]` | 11 | 16 |
| 12 | `put("key8", 8)`: `i=12`, empty bucket → **APPEND** | `[('key8', 8)]` | 12 | 16 |
| 13 | `put("key9", 9)`: `i=2` — same bucket as `"carol"`/`"key4"` → **COLLISION**, append; `size=13`, `13/16=0.8125 > 0.75` → **RESIZE** to 32 | `[('carol', 40), ('key4', 4), ('key9', 9)]` (pre-resize) | 13 | 32 |
| 14 | `put("key10", 10)`: `i=24` (cap 32) — same bucket as `"key2"`/`"key7"` → **COLLISION**, append | `[('key2', 2), ('key7', 7), ('key10', 10)]` | 14 | 32 |
| 15 | `put("key11", 11)`: `i=3`, empty bucket → **APPEND** | `[('key11', 11)]` | 15 | 32 |
| 16 | `put("key12", 12)`: `i=31`, empty bucket → **APPEND** | `[('key12', 12)]` | 16 | 32 |
| 17 | `put("key13", 13)`: `i=1` — same bucket as `"alice"` → **COLLISION**, append | `[('alice', 30), ('key13', 13)]` | 17 | 32 |
| 18 | `put("key14", 14)`: `i=24` — same bucket → **COLLISION**, append | `[('key2', 2), ('key7', 7), ('key10', 10), ('key14', 14)]` | 18 | 32 |
| 19 | `put("key15", 15)`: `i=22`, empty bucket → **APPEND** | `[('key15', 15)]` | 19 | 32 |
| 20 | `put("key16", 16)`: `i=28` — same bucket as `"key8"` → **COLLISION**, append | `[('key8', 8), ('key16', 16)]` | 20 | 32 |
| 21 | `put("key17", 17)`: `i=31` — same bucket as `"key12"` → **COLLISION**, append | `[('key12', 12), ('key17', 17)]` | 21 | 32 |
| 22 | `put("key18", 18)`: `i=24` — same bucket → **COLLISION**, append | `[('key2', 2), ('key7', 7), ('key10', 10), ('key14', 14), ('key18', 18)]` | 22 | 32 |
| 23 | `put("key19", 19)`: `i=7`, empty bucket → **APPEND**; `size=23`, `23/32=0.719 ≤ 0.75` → no resize | `[('key19', 19)]` | 23 | 32 |

Final state after this loop: `size=23`, `capacity=32` — matches `assert ht.capacity > 4`. Row 4 and row 7 and row 13 are the rows that matter: each is the insert that pushed the load factor past `0.75` and fired `_resize()`, which is why the demo's `capacity` column jumps `4 → 8 → 16 → 32` instead of climbing by one.

#### `get(key, default=None)` — same index, same scan, no mutation

1. Compute `i = self._index(key)` exactly as `put` does.
2. Linearly scan `self.buckets[i]` for a matching key.
3. Return its value if found, else return `default`.

Demo: `ht.get("alice")`, `ht.get("dave", "not found")` (both before the growth loop), then `ht.get("alice") == 30` and `ht.get("key15") == 15` (both after the growth loop, so `capacity=32`).

| Step | Action | `bucket[i]` inspected | Returns |
|---|---|---|---|
| 1 | `get("alice")`: `i=1` (cap 4) → scan, match `("alice", 30)` → **RECORD** | `[('alice', 30)]` | `30` |
| 2 | `get("dave", "not found")`: `i=3` (cap 4) → bucket empty, no match → **RECORD** default | `[]` | `"not found"` |
| 3 | `assert ht.get("alice") == 30` → **PASS** | `[('alice', 30)]` | `30` |
| 4 | `assert ht.get("dave", "not found") == "not found"` → **PASS** | `[]` | `"not found"` |
| 5 | `assert ht.get("alice") == 30` (post-growth, cap 32, `i=1`) → **PASS** | `[('alice', 30), ('key13', 13)]` | `30` |
| 6 | `assert ht.get("key15") == 15` (post-growth, cap 32, `i=22`) → **PASS** | `[('key15', 15)]` | `15` |

Final two rows match `assert ht.get("alice") == 30` and `assert ht.get("key15") == 15` printed right after `print("capacity after growth:", ht.capacity)`. Row 5 is the one that matters: `"alice"` was inserted back when `capacity=4`, survived two full rehashes, and `get` still finds it purely because `_resize` re-`put` every old entry under the new capacity — nothing about `"alice"`'s bucket position is preserved across a resize.

#### `_resize()` — double capacity, rebuild empty buckets, re-`put` everything

1. Save the old bucket list (`old_buckets`), double `self.capacity`, replace `self.buckets` with a fresh list of empty lists at the new size, and reset `self.size = 0`.
2. Walk `old_buckets` bucket by bucket, entry by entry (in existing list order), and call `self.put(k, v)` for each — this recomputes each key's index under the new (larger) `capacity` and re-increments `size`, so every entry effectively "moves."

Demo: the first resize, triggered by row 4 of the `put` trace above (`ht.put("key0", 0)` pushes `size/capacity` to `4/4`), doubles capacity from 4 to 8.

| Step | Action | `bucket[i]` after (cap 8) | `size` after |
|---|---|---|---|
| 1 | `capacity` doubled `4 → 8`; `buckets` rebuilt as 8 empty lists; `size` reset to `0` | — | 0 |
| 2 | re-`put("alice", 30)`: `i=1` → **APPEND** | `[('alice', 30)]` | 1 |
| 3 | re-`put("key0", 0)`: `i=5` → **APPEND** | `[('key0', 0)]` | 2 |
| 4 | re-`put("bob", 25)`: `i=2` → **APPEND** | `[('bob', 25)]` | 3 |
| 5 | re-`put("carol", 40)`: `i=2` — same bucket as `"bob"` → **COLLISION**, append | `[('bob', 25), ('carol', 40)]` | 4 |

After this resize, `size=4`, `capacity=8` — exactly the values row 4 of the `put` table reports. The other two resizes in the demo (triggered by `"key3"`, doubling `8 → 16`, and by `"key9"`, doubling `16 → 32`) run the identical mechanic on 7 and 13 entries respectively; you can see their effect directly in the `put` table above — e.g. `"carol"` and `"key9"` end up sharing bucket 2 again after the third resize, and `"key2"`/`"key7"` land together in bucket 24.

#### Mental model

- `_index` is the *only* place `capacity` matters — `put`, `get`, and the rehashing loop inside `_resize` all call it fresh, so the same key can (and does) live at a different index before and after a resize.
- A "collision" in this design is not an error path — it is simply two `(key, value)` pairs sitting in the same bucket's Python list; correctness comes from `put`'s linear scan checking every entry in the bucket, not from the index alone.
- The `0.75` load-factor check in `put` is what keeps average bucket length bounded — without it, buckets could grow arbitrarily long and every `get`/`put` would degrade toward `O(n)`.
- `_resize` never inspects a value — it re-derives every key's home purely from `_index`, which is why it must fully rebuild `buckets` and replay every stored `(key, value)` through `put` rather than trying to shuffle entries in place.
- The general lesson: any hash-table variant (open addressing, robin hood, etc.) still needs this same triangle — a deterministic index function, an explicit collision-resolution rule inside lookup/insert, and a resize trigger tied to load factor — the specific data structure per bucket is just an implementation choice.

## 2. Collisions: same bucket, different keys
Demonstrate two keys landing in the same bucket and both being retrievable.

In [ ]:
tiny = SimpleHashTable(capacity=8)
# For an int key, Python's hash(x) == x, so keys 3 and 11 BOTH land in bucket (x % 8) == 3.
tiny.put(3, "three")
tiny.put(11, "eleven")     # guaranteed collision with 3 -- same bucket, different key
assert tiny._index(3) == tiny._index(11)          # confirm they really do collide
# Both keys are still retrieved correctly -- chaining handles the collision transparently.
assert tiny.get(3) == "three" and tiny.get(11) == "eleven"
bucket_sizes = [len(b) for b in tiny.buckets]
print("bucket sizes:", bucket_sizes, "(the bucket at index", tiny._index(3), "holds both colliding keys)")

## 3. Python's `dict` / `set` / `Counter`

In [ ]:
from collections import defaultdict, Counter

d = {}
d["alice"] = 30
assert d.get("alice") == 30
assert d.get("missing", 0) == 0
assert "alice" in d
del d["alice"]
assert "alice" not in d

freq = defaultdict(int)                 # auto-creates missing keys with a default value
for ch in "abracadabra":
    freq[ch] += 1
print("letter frequency:", dict(freq))
assert freq["a"] == 5

counts = Counter(["a", "b", "a", "c", "a"])
print("Counter:", counts)
assert counts.most_common(1) == [("a", 3)]

seen = set()
seen.add("x")
assert "x" in seen and "y" not in seen

## 4. Classic use -- Two Sum via complement lookup (O(n), not O(n^2))

In [ ]:
def two_sum(nums, target):
    """For each number, check if its complement was already seen -- O(1) average per check."""
    seen = {}                                    # value -> index
    for i, x in enumerate(nums):
        need = target - x
        if need in seen:                          # O(1) average membership test
            return [seen[need], i]
        seen[x] = i
    return []

result = two_sum([2, 7, 11, 15], 9)
print("two_sum([2,7,11,15], 9) ->", result)
assert result == [0, 1]

## 5. Classic use -- Group Anagrams (grouping by a derived key)

In [ ]:
from collections import defaultdict

def group_anagrams(words):
    """Words that are anagrams of each other share the same SORTED-LETTERS key."""
    groups = defaultdict(list)
    for w in words:
        key = "".join(sorted(w))                  # anagrams collapse to the same key
        groups[key].append(w)
    return list(groups.values())

result = group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"])
print("groups:", result)
# Order of groups/items isn't guaranteed -- check membership instead.
result_sets = [set(g) for g in result]
assert {"eat", "tea", "ate"} in result_sets
assert {"tan", "nat"} in result_sets
assert {"bat"} in result_sets

## ✅ Recap
- A hash table computes an **index** from the key instead of searching for it -- `O(1)` average get/put/delete.
- Collisions are unavoidable (pigeonhole principle) -- handled via **chaining** or **open addressing**.
- Resizing when the load factor climbs keeps operations `O(1)` **amortized**, even though a single resize is `O(n)`.
- Python's `dict`/`set`/`Counter`/`defaultdict` are production-grade hash tables -- reach for them first.
- Classic uses: complement lookup (Two Sum), frequency counting, grouping by a derived key, memoization.

Next: [`17_Sorting_Algorithms`](../17_Sorting_Algorithms/README.md).